# Direct Multi-Horizon Forecasting vs. Autoregressive — XGBoost only

Compares two forecasting strategies on rack R0605-PA (lookback=60, horizon=30):

- **Autoregressive (current pipeline, unchanged)**: predicts one step, feeds the
  full predicted feature vector back in, repeats `horizon` times. Errors compound.
- **Direct (new, notebook-only)**: predicts all `horizon` steps of the TARGET
  column in a single forward pass. No feedback loop, no compounding error. Since
  nothing needs to be fed back in, it only needs to predict the target -- not the
  full multi-output feature vector the autoregressive path requires.

Temporary/exploratory notebook -- no changes to `rack_forecast/` or
`scripts/run_prediction.py`. Only XGBoost is used (fastest model) for both.

## 0. Setup

In [1]:
import time
import numpy as np
import pandas as pd

from rack_forecast import ExperimentConfig
from rack_forecast.data import build_dataset
from rack_forecast.pipeline import prepare_data
from rack_forecast.windowing import make_supervised
from rack_forecast.trainer import build_and_train, DEVICE
from rack_forecast.evaluate import evaluate, compute_metrics, _inverse_target
from rack_forecast.models.xgboost import XGBoostModel

print('Device:', DEVICE)

[rack_forecast] device: cuda
Device: cuda


## 1. Config & data

Same config style as `feature_eda.ipynb`: `train_days=60` for fast iteration.
Both methods below share this exact `data`/`prep` -- same train/test split, same
scaler, same input features. Only the forecasting STRATEGY differs.

In [2]:
cfg = ExperimentConfig(
    target_rack='R0605-PA',
    lookback=60,
    horizon=30,
    models=['xgboost'],
    fast_mode=True,
    train_days=60,
    predict_days=None,
    run_id='direct_forecast_eda',
)
print(cfg)

data = build_dataset(cfg)
target = cfg.target_col
print('Base dataset:', data.shape, data.index.min(), '->', data.index.max())

prep = prepare_data(data, cfg)
print(f'Train: {len(prep.X_train):,}  Test: {len(prep.X_test):,}  Features: {prep.n_feat}')
print('target_idx:', prep.target_idx, '->', prep.feature_cols[prep.target_idx])

ExperimentConfig(target_rack='R0605-PA', resample='30s', lookback=60, horizon=30, models=['xgboost'], dl_epochs=30, fast_mode=True, train_days=60, predict_days=None, run_id='direct_forecast_eda')


Base dataset: (521246, 17) 2022-09-01 00:00:00 -> 2023-02-28 23:59:30


Train: 172,766  Test: 80,640  Features: 17
target_idx: 0 -> R0605-PA__kW


## 2. Autoregressive baseline (current pipeline, UNCHANGED)

Exactly today's approach: `make_supervised` -> `build_and_train('xgboost')` ->
`evaluate()` (autoregressive rollout) -> `compute_metrics()`.

In [3]:
t0 = time.time()
X_3d_auto, y_sup_auto = make_supervised(prep.X_train, prep.y_train, cfg.lookback)
print('Autoregressive train windows:', X_3d_auto.shape)

model_auto = build_and_train('xgboost', X_3d_auto, y_sup_auto, cfg.lookback,
                              prep.n_feat, dl_epochs=cfg.dl_epochs)
train_time_auto = time.time() - t0

t0 = time.time()
preds_auto, actuals_auto = evaluate(model_auto, prep.X_test, prep.y_test, cfg.lookback,
                                    cfg.horizon, prep.target_idx, prep.scaler, prep.n_feat)
infer_time_auto = time.time() - t0

n_windows_auto = len(preds_auto) // cfg.horizon
metrics_auto = compute_metrics(actuals_auto, preds_auto)
print(f'n_windows={n_windows_auto}  train={train_time_auto:.1f}s  infer={infer_time_auto:.1f}s')
print(f"MAE={metrics_auto['MAE']:.4f}  RMSE={metrics_auto['RMSE']:.4f}  "
      f"MAPE={metrics_auto['MAPE(%)']:.2f}%  R2={metrics_auto['R2']:.4f}")

Autoregressive train windows: (172706, 60, 17)
    backend: cuda


  steps:   0%|          | 0/30 [00:00<?, ?step/s]

C:\Users\quang\anaconda3\envs\ntu_cooling\Lib\site-packages\xgboost\core.py:751: UserWarning: [14:30:57] WARNING: C:\Users\task_177929407859648\croot\xgboost-split_1779294268734\work\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  steps:   3%|▎         | 1/30 [00:00<00:12,  2.38step/s]

  steps:  10%|█         | 3/30 [00:00<00:04,  6.18step/s]

...

  steps:  77%|███████▋  | 23/30 [00:02<00:00, 12.29step/s]

  steps:  83%|████████▎ | 25/30 [00:02<00:00, 12.20step/s]

  steps:  90%|█████████ | 27/30 [00:02<00:00, 12.48step/s]

  steps:  97%|█████████▋| 29/30 [00:02<00:00, 12.78step/s]

n_windows=2686  train=93.6s  infer=2.7s
MAE=0.0121  RMSE=0.0159  MAPE=2.48%  R2=0.3128


## 3. Direct multi-horizon forecasting (new, notebook-only code)

- Training windows: sliding/overlapping (stride 1, standard for training -- more
  samples), target-only `horizon`-length `y` (not the next single step, not other
  features).
- Test windows: mirrors `evaluate()`'s EXACT non-overlapping window definition
  (`n_windows = (len(X_test) - lookback) // horizon`, same seed/target slicing)
  so both methods are scored on identical test windows.
- Reuses the same `XGBoostModel` class -- it's generic to output width, so fitting
  it with a `(n, horizon)` target instead of `(n, n_feat)` just works.

In [4]:
def make_direct_train_windows(X_2d, target_idx, lookback, horizon):
    """Overlapping training windows: X = past `lookback` steps (all features),
    y = next `horizon` steps of the TARGET column only."""
    n = len(X_2d)
    Xs, ys = [], []
    for i in range(lookback, n - horizon + 1):
        Xs.append(X_2d[i - lookback:i])
        ys.append(X_2d[i:i + horizon, target_idx])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)


def make_direct_test_windows(X_2d, target_idx, lookback, horizon):
    """Mirrors evaluate()'s exact non-overlapping test-window definition so both
    methods are scored on identical windows."""
    n_windows = (len(X_2d) - lookback) // horizon
    Xs = np.stack([
        X_2d[w * horizon: w * horizon + lookback]
        for w in range(n_windows)
    ], axis=0).astype(np.float32)
    ys = np.stack([
        X_2d[lookback + w * horizon: lookback + w * horizon + horizon, target_idx]
        for w in range(n_windows)
    ], axis=0).astype(np.float32)
    return Xs, ys, n_windows

In [5]:
t0 = time.time()
X_3d_direct, y_direct = make_direct_train_windows(prep.X_train, prep.target_idx,
                                                   cfg.lookback, cfg.horizon)
print('Direct train windows:', X_3d_direct.shape, '-> y:', y_direct.shape)

model_direct = XGBoostModel()
model_direct.fit(X_3d_direct.reshape(len(X_3d_direct), -1), y_direct)
train_time_direct = time.time() - t0

t0 = time.time()
X_test_direct, y_test_direct, n_windows_direct = make_direct_test_windows(
    prep.X_test, prep.target_idx, cfg.lookback, cfg.horizon)
preds_scaled = model_direct.predict_batch(X_test_direct)          # (n_windows, horizon), one shot

preds_direct = _inverse_target(preds_scaled.ravel(), prep.scaler, prep.target_idx, prep.n_feat)
actuals_direct = _inverse_target(y_test_direct.ravel(), prep.scaler, prep.target_idx, prep.n_feat)
infer_time_direct = time.time() - t0

metrics_direct = compute_metrics(actuals_direct, preds_direct)
print(f'n_windows={n_windows_direct}  train={train_time_direct:.1f}s  infer={infer_time_direct:.1f}s')
print(f"MAE={metrics_direct['MAE']:.4f}  RMSE={metrics_direct['RMSE']:.4f}  "
      f"MAPE={metrics_direct['MAPE(%)']:.2f}%  R2={metrics_direct['R2']:.4f}")

assert n_windows_direct == n_windows_auto, \
    f'Window count mismatch: direct={n_windows_direct} vs autoregressive={n_windows_auto}'
print('n_windows match confirmed -- both methods scored on identical test windows.')

Direct train windows: (172677, 60, 17) -> y: (172677, 30)


n_windows=2686  train=144.3s  infer=0.6s
MAE=0.0120  RMSE=0.0158  MAPE=2.47%  R2=0.3232
n_windows match confirmed -- both methods scored on identical test windows.


## 4. Comparison table

In [6]:
comp = pd.DataFrame([
    {'strategy': 'autoregressive_xgboost', 'n_windows': n_windows_auto,
     **metrics_auto, 'train_s': round(train_time_auto, 1), 'infer_s': round(infer_time_auto, 1)},
    {'strategy': 'direct_xgboost', 'n_windows': n_windows_direct,
     **metrics_direct, 'train_s': round(train_time_direct, 1), 'infer_s': round(infer_time_direct, 1)},
]).sort_values('RMSE').reset_index(drop=True)
comp

,strategy,n_windows,MAE,RMSE,MAPE(%),R2,train_s,infer_s
0,direct_xgboost,2686,0.012046,0.015820,2.471327,0.323248,144.3,0.6
1,autoregressive_xgboost,2686,0.012093,0.015942,2.476792,0.312757,93.6,2.7


## 5. Takeaway

In [7]:
winner = comp.iloc[0]
loser = comp.iloc[1]
pct = 100 * (loser['RMSE'] - winner['RMSE']) / winner['RMSE']
print(f"Winner: {winner['strategy']}  RMSE={winner['RMSE']:.4f}  "
      f"(vs {loser['strategy']} RMSE={loser['RMSE']:.4f}, {pct:.1f}% worse)")

Winner: direct_xgboost  RMSE=0.0158  (vs autoregressive_xgboost RMSE=0.0159, 0.8% worse)


**Scope caveat**: single rack (R0605-PA), single model (XGBoost), single
horizon config (lookback=60/horizon=30), `train_days=60`-clipped. A win here is a
signal about which STRATEGY suits this data/horizon combination -- not a final
verdict, and not yet tested on other horizons, racks, or model types (direct
forecasting for the DL models would need an output-layer size change, unlike
XGBoost which just works with whatever `y` width it's fit on).